<a href="https://colab.research.google.com/github/Reben80/Data110-21843--fall25/blob/main/Week14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CountVectorizer from `sklearn.feature_extraction.text` transforms a collection of text documents into a numerical matrix by counting how often each word appears. This creates a structured document–word matrix that makes it easy to analyze patterns in the text or visualize them, as we do in the example below.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import matplotlib.pyplot as plt

Write a simple documents "corpus" that have similar words

In [ ]:
# 1. Sample “corpus” of short documents
documents = [
    "cats and dogs are friends and ",
    "cats love milk and fish",
    "dogs love bones and meat",
    "fish live in water",
]

# 2. Turn text into a document–term matrix


vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)   # rows = docs, columns = words
words = vectorizer.get_feature_names_out()

df = pd.DataFrame(X.toarray(), columns=words)
print(df)
df.info()

A matrix or pixel-based text visualization turns text into a grid of colored cells, where each cell represents a numerical property such as word frequency. This helps us quickly spot patterns across documents that would be hard to see by reading alone. The heatmap below shows how often each word appears in each document.

In [ ]:
plt.figure(figsize=(8, 4))
plt.imshow(df.values, aspect="auto")
plt.colorbar(label="Word count")
plt.xticks(range(len(words)), words, rotation=90)
plt.yticks(range(len(documents)), [f"doc{i+1}" for i in range(len(documents))])
plt.title("Document–Word Matrix (Pixel / Matrix Text Visualization)")
plt.tight_layout()
plt.show()


In [ ]:
!pip install gensim

This section trains a small Word2Vec model on our sample sentences. `Word2Vec` learns a vector (embedding) for each word based on how it appears in context.

-`vector_size=50` sets the length of each word’s embedding.
-`window=3` means the model looks at up to 3 words on each side for context.
-`min_count=1` keeps all words, even those appearing only once.
-`epochs=200` gives the model enough passes over the data to learn meaningful relationships.

In [ ]:
# Install gensim if needed (uncomment in Colab)
# !pip install gensim

from gensim.models import Word2Vec
from sklearn.decomposition import PCA


# 1. Tiny toy corpus (for demo)
sentences = [
    ["cat", "dog", "pet", "home"],
    ["cat", "kitten", "purr"],
    ["dog", "puppy", "bark"],
    ["lion", "tiger", "wild", "animal"],
    ["paris", "france", "city", "capital"],
    ["berlin", "germany", "city", "capital"],
    ["madrid", "spain", "city", "capital"],
    ["happy", "joy", "smile"],
    ["sad", "cry", "tears"],
]

# 2. Train a small Word2Vec model
model = Word2Vec(
    sentences,
    vector_size=50,   # embedding dimension
    window=3,
    min_count=1,
    workers=1,
    epochs=200
)

# 3. Pick some words to visualize
words = [
    "cat", "dog", "kitten", "puppy",
    "lion", "tiger",
    "paris", "france", "berlin", "germany", "madrid", "spain",
    "happy", "joy", "sad", "cry"
]

# Filter only words present in the model (just in case)
words = [w for w in words if w in model.wv.key_to_index]

# 4. Get their embeddings
vectors = [model.wv[w] for w in words]

# 5. Reduce to 2D using PCA
#Principal Component Analysis
pca = PCA(n_components=2)
# n_components=2 tells PCA to reduce the data down to 2 principal components, which means 2 dimensions. python is not good for 3D viz
points_2d = pca.fit_transform(vectors)

x=points_2d[:, 0]
y=points_2d[:, 1]

# 6. Plot them
plt.figure(figsize=(10, 8))
plt.scatter(x, y, s=80)

for i, word in enumerate(words):
    plt.annotate(
        word,
        (x[i], y[i]),
        textcoords="offset points",
        xytext=(5, 5),      # moves text away from the dot
        ha='left',
        fontsize=12,
        weight='bold'
    )

plt.title("Word Embedding Space (2D PCA Projection)", fontsize=16)
plt.xlabel("PC 1", fontsize=14)
plt.ylabel("PC 2", fontsize=14)
plt.grid(True)
plt.show()